In [ ]:
import sys
sys.path.insert(0, '../lib')

import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import anndata as ad
import seaborn as sns
import warnings

import common_data

In [3]:
%config InlineBackend.figure_format = "retina"

In [ ]:
list_of_libs = pd.read_csv(common_data.SC_LIBRARIES, index_col=0)

In [20]:
list_of_libs.groupby('name').individual.nunique()

name
AUXORA205       4
Duke_ozone      7
PASC           25
SCRIPT        266
SSc-ILD         2
Name: individual, dtype: int64

In [21]:
list_of_libs.groupby('name').library_id.nunique()

name
AUXORA205       6
Duke_ozone      8
PASC           25
SCRIPT        298
SSc-ILD         3
Name: library_id, dtype: int64

In [ ]:
datasets = []
gene_counts = []
base_dir = common_data.SC_CELLRANGER
for _, row in list_of_libs.iterrows():
    lib = row.library_id
    lib_file = os.path.join(base_dir, lib, "filtered_feature_bc_matrix.h5")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ds = sc.read_10x_h5(lib_file)
        ds.var_names_make_unique()
    if ds.var_names[-1].startswith("GRCh38______"):
        ds.var_names = ds.var_names.str.replace("GRCh38______", "").str.replace(
            "^SARS-CoV-2i_", "SARS-CoV-2_", regex=True
        )
    elif ds.var_names[-1].startswith("GRCh38_"):
        ds.var_names = ds.var_names.str.replace("GRCh38_", "").str.replace(
            "^SARS2\_\_", "SARS-CoV-2_", regex=True
        )
    for c in list_of_libs.columns:
        ds.obs[c] = row[c]
    ds.obs_names = row.library_id + "_" + ds.obs_names.str.replace("-\d+$", "", regex=True)
    gene_counts.append(ds.var_names.size)
    datasets.append(ds)
    print(".", end="", flush=True)

....................................................................................................................................................................................................................................................................................................................................................

Without `merge=same` we lose ensembl ids

In [48]:
ds = ad.concat(datasets, join='outer', merge='same')

In [49]:
ds.obs.groupby('name').individual.nunique()

name
AUXORA205       4
Duke_ozone      7
PASC           25
SCRIPT        266
SSc-ILD         2
Name: individual, dtype: int64

In [50]:
ds.obs.groupby('name').library_id.nunique()

name
AUXORA205       6
Duke_ozone      8
PASC           25
SCRIPT        298
SSc-ILD         3
Name: library_id, dtype: int64

In [51]:
ds.var["mito"] = ds.var_names.str.startswith("MT-")

In [52]:
ds.var["ribo"] = ds.var_names.str.match("^RP(L|S)")

In [53]:
sc.pp.calculate_qc_metrics(
    ds,
    qc_vars=["mito", "ribo"],
    percent_top=[10, 20],
    log1p=False,
    inplace=True
)

In [ ]:
ds.write_h5ad(common_data._sc_root / '01a_raw.h5ad')

In [56]:
pd.Series(gene_counts).describe()

count      340.0
mean     33550.0
std          0.0
min      33550.0
25%      33550.0
50%      33550.0
75%      33550.0
max      33550.0
dtype: float64

In [55]:
ds.var

,gene_ids,feature_types,genome,mito,ribo,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
SARS-CoV-2_ORF1ab,SARS-CoV-2i_gene-GU280_gp01,Gene Expression,SARS-CoV-2i,False,False,19029,0.038084,99.370445,115112.0
SARS-CoV-2_S,SARS-CoV-2i_gene-GU280_gp02,Gene Expression,SARS-CoV-2i,False,False,8599,0.009752,99.715511,29475.0
SARS-CoV-2_ORF3a,SARS-CoV-2i_gene-GU280_gp03,Gene Expression,SARS-CoV-2i,False,False,4321,0.003539,99.857044,10697.0
SARS-CoV-2_E,SARS-CoV-2i_gene-GU280_gp04,Gene Expression,SARS-CoV-2i,False,False,1244,0.000684,99.958844,2068.0
SARS-CoV-2_M,SARS-CoV-2i_gene-GU280_gp05,Gene Expression,SARS-CoV-2i,False,False,6241,0.006620,99.793523,20011.0
...,...,...,...,...,...,...,...,...,...
AC233755.2,GRCh38______ENSG00000277856,Gene Expression,GRCh38,False,False,4635,0.215264,99.846656,650660.0
AC233755.1,GRCh38______ENSG00000275063,Gene Expression,GRCh38,False,False,11563,0.500670,99.617450,1513329.0
AC240274.1,GRCh38______ENSG00000271254,Gene Expression,GRCh38,False,False,34768,0.011878,98.849735,35904.0
AC213203.1,GRCh38______ENSG00000277475,Gene Expression,GRCh38,False,False,8,0.000003,99.999735,8.0
